# 🚀 Fine-tuning Qwen/Qwen2.5-Coder-7B-Instruct cho NL to FOL Translation (V2)

Notebook này thực hiện huấn luyện Fine-Tuning (SFT) mô hình **Qwen/Qwen2.5-Coder-7B-Instruct** để dịch Natural Language sang First-Order Logic (FOL).

## 1. Cài đặt Thư viện

In [1]:
!pip install -q transformers peft trl accelerate datasets evaluate huggingface_hub z3-solver


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


## 2. Khai báo Thư viện & Kiểm tra GPU

In [2]:
import os
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer

# Khai báo import linh hoạt cho DataCollatorForCompletionOnlyLM
try:
    from trl import DataCollatorForCompletionOnlyLM
except ImportError:
    try:
        from trl.trainer import DataCollatorForCompletionOnlyLM
    except ImportError:
        try:
            from trl.trainer.utils import DataCollatorForCompletionOnlyLM
        except ImportError:
            from transformers import DataCollatorForLanguageModeling
            
            class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
                def __init__(self, response_template, tokenizer, *args, **kwargs):
                    kwargs.setdefault('mlm', False)
                    super().__init__(tokenizer=tokenizer, *args, **kwargs)
                    self.response_template = response_template
                    self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)
                    
                def torch_call(self, examples):
                    batch = super().torch_call(examples)
                    response_token_ids = self.response_token_ids
                    n = len(response_token_ids)
                    
                    labels = batch['labels'].clone()
                    for i in range(len(examples)):
                        input_ids = batch['input_ids'][i].tolist()
                        idx = -1
                        for j in range(len(input_ids) - n + 1):
                            if input_ids[j:j+n] == response_token_ids:
                                idx = j + n
                                break
                        if idx != -1:
                            labels[i, :idx] = -100
                    batch['labels'] = labels
                    return batch

# Kiểm tra GPU
print("CUDA/ROCm Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("VRAM Memory Total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

CUDA/ROCm Available: True
GPU Device Name: 
VRAM Memory Total (GB): 191.69


## 3. Cấu hình Đường dẫn & Nạp Tập dữ liệu

In [8]:
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATA_FILE = "./data/qwen_logic_sft.jsonl"
OUTPUT_DIR = "./qwen_logic_v2_lora"
max_seq_length = 2048

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Không tìm thấy file tập dữ liệu {DATA_FILE}!")

print("📦 Đang tải tập dữ liệu Logic SFT...")
dataset = load_dataset("json", data_files={"train": DATA_FILE}, split="train")
print(f"Đã tải thành công {len(dataset)} mẫu dữ liệu!")

📦 Đang tải tập dữ liệu Logic SFT...


Generating train split: 0 examples [00:00, ? examples/s]

Đã tải thành công 1686 mẫu dữ liệu!


## 4. Khởi tạo Tokenizer & Format Dataset

In [9]:
print("⏳ Đang khởi tạo Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_and_tokenize(example):
    output_texts = []
    for msg_list in example['messages']:
        text = tokenizer.apply_chat_template(msg_list, tokenize=False)
        output_texts.append(text)
    return tokenizer(output_texts, truncation=True, max_length=max_seq_length)

print("📝 Đang tokenize dataset...")
dataset = dataset.map(format_and_tokenize, batched=True, remove_columns=dataset.column_names)
print("Hoàn thành!")

⏳ Đang khởi tạo Tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

📝 Đang tokenize dataset...


Map:   0%|          | 0/1686 [00:00<?, ? examples/s]

Hoàn thành!


## 5. Phân chia Train / Val Split

In [10]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"📊 Train samples: {len(train_dataset)}")
print(f"📊 Val samples: {len(val_dataset)}")

📊 Train samples: 1517
📊 Val samples: 169


## 6. Tải Mô hình Qwen/Qwen2.5-Coder-7B-Instruct

In [11]:
print(f"🔥 Đang tải mô hình {BASE_MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
model.enable_input_require_grads()
model.config.use_cache = False
print("Tải mô hình thành công!")

🔥 Đang tải mô hình Qwen/Qwen2.5-Coder-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tải mô hình thành công!


## 7. Thiết lập LoRA (Tối ưu cho Reasoning/Z3)

In [12]:
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


## 8. Cấu hình Trainer & SFTTrainer

In [13]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    per_device_train_batch_size=4,         
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=10,
    num_train_epochs=8,
    bf16=True,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    optim="adamw_torch",
    warmup_ratio=0.05,
    weight_decay=0.01,
    report_to="none"
)

response_template = "<|im_start|>assistant\n"
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template, 
    tokenizer=tokenizer, 
    mlm=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    data_collator=data_collator
)
print("✅ Sẵn sàng huấn luyện!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


✅ Sẵn sàng huấn luyện!


## 9. Bắt đầu Train

In [14]:
print("🚀 BẤM NÚT FINE-TUNING...")
trainer.train()
print("🎉 Hoàn tất huấn luyện!")

# Lưu mô hình LoRA
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Đã lưu adapter tại: {OUTPUT_DIR}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 BẤM NÚT FINE-TUNING...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.155700,0.147268,0.155088,1091293.000000,0.955326
2,0.019900,0.022404,0.030013,2182586.000000,0.993498
3,0.005400,0.006824,0.009925,3273879.000000,0.998201
4,0.002100,0.004940,0.005588,4365172.000000,0.998865
5,0.001600,0.002708,0.003718,5456465.000000,0.999335
6,0.001300,0.002910,0.002947,6547758.000000,0.999351
7,0.000900,0.002452,0.002578,7639051.000000,0.999471
8,0.000900,0.002543,0.002430,8730344.000000,0.999471


🎉 Hoàn tất huấn luyện!
Đã lưu adapter tại: ./qwen_2.5_coder_7b_logic_lora


In [18]:
# ## 10. Đánh giá độ chính xác (Accuracy Evaluation) sau SFT
#
# Cell này thực hiện chạy thử nghiệm mô hình đã SFT trên tập Validation (`val_dataset`), trích xuất mã Z3 Python, thực thi độc lập và đối chiếu kết quả với đáp án chuẩn.

import re
import subprocess
import sys
from tqdm import tqdm

def execute_z3_code_with_helpers(code_str):
    import re
    
    # Tự động sửa lỗi mất cân bằng dấu ngoặc đơn (Parentheses Balancer)
    def fix_parentheses(code):
        fixed_lines = []
        for line in code.splitlines():
            open_cnt = 0
            close_cnt = 0
            in_string = False
            string_char = None
            for char in line:
                if char in ["'", '"']:
                    if not in_string:
                        in_string = True
                        string_char = char
                    elif string_char == char:
                        in_string = False
                elif not in_string:
                    if char == '(':
                        open_cnt += 1
                    elif char == ')':
                        close_cnt += 1
            if open_cnt > close_cnt:
                line += ')' * (open_cnt - close_cnt)
            elif close_cnt > open_cnt:
                diff = close_cnt - open_cnt
                stripped_line = line.rstrip()
                if stripped_line.endswith(')' * diff):
                    line = stripped_line[:-diff]
                else:
                    parts = list(line)
                    removed = 0
                    for idx in range(len(parts) - 1, -1, -1):
                        if parts[idx] == ')' and removed < diff:
                            parts.pop(idx)
                            removed += 1
                    line = "".join(parts)
            fixed_lines.append(line)
        return "\n".join(fixed_lines)

    code_str = fix_parentheses(code_str)
    
    # Tự động sửa lỗi cú pháp >> thành Implies
    code_str = re.sub(r'([A-Za-z0-9_]+)\s*>>\s*([A-Za-z0-9_]+)', r'Implies(\1, \2)', code_str)
    
    env_code = """
import z3
from z3 import *

def check_property(solver, prop):
    solver.push()
    solver.add(z3.Not(prop))
    is_always_true = (solver.check() == z3.unsat)
    solver.pop()
    solver.push()
    solver.add(prop)
    is_always_false = (solver.check() == z3.unsat)
    solver.pop()
    if is_always_true:
        return 'Yes'
    elif is_always_false:
        return 'No'
    else:
        return 'Unknown'

def check_mcq(solver, options_dict):
    yes_opts = []
    for label, prop in options_dict.items():
        solver.push()
        solver.add(z3.Not(prop))
        is_always_true = (solver.check() == z3.unsat)
        solver.pop()
        if is_always_true:
            yes_opts.append(label)
    if len(yes_opts) == 1:
        return yes_opts[0]
    no_opts = []
    for label, prop in options_dict.items():
        solver.push()
        solver.add(prop)
        is_always_false = (solver.check() == z3.unsat)
        solver.pop()
        if is_always_false:
            no_opts.append(label)
    if len(yes_opts) == 0 and len(no_opts) == 1:
        return no_opts[0]
    if len(yes_opts) > 1:
        return yes_opts[0]
    return 'Unknown'
""" + "\n" + code_str

    try:
        res = subprocess.run([sys.executable, "-c", env_code], capture_output=True, text=True, timeout=5)
        output = res.stdout.strip()
        m = re.search(r'Answer:\s*(.*)', output, re.IGNORECASE)
        ans = m.group(1).strip() if m else "NoOutput"
        return ans, res.stderr.strip()
    except Exception as e:
        return "Error", str(e)

def extract_python_code(text):
    if "```python" in text:
        return text.split("```python")[1].split("```")[0].strip()
    elif "```" in text:
        return text.split("```")[1].split("```")[0].strip()
    return ""

def evaluate_model_accuracy(model, tokenizer, test_jsonl_file, num_samples=50):
    # Load original validation items to get ground truth and user prompt
    val_items = []
    with open(test_jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            val_items.append(json.loads(line))
            
    # Take a subset for evaluation if specified
    import random
    random.seed(42)
    if len(val_items) > num_samples:
        val_items = random.sample(val_items, num_samples)
        
    print(f"📊 Đang chạy đánh giá trên {len(val_items)} mẫu test ngẫu nhiên...")
    
    correct = 0
    total = 0
    
    # Temporarily set model to eval mode
    model.eval()
    
    for item in tqdm(val_items):
        messages = item["messages"]
        user_prompt = messages[1]["content"]
        assistant_expected = messages[2]["content"]
        
        # Extract expected answer
        z3_code_expected = extract_python_code(assistant_expected)
        expected_ans, _ = execute_z3_code_with_helpers(z3_code_expected)
        
        # Format input using chat template
        input_messages = [
            {"role": "system", "content": messages[0]["content"]},
            {"role": "user", "content": user_prompt}
        ]
        
        input_ids = tokenizer.apply_chat_template(input_messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_new_tokens=2048,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            
        generated_ids = outputs[0][len(input_ids[0]):]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        
        # Extract code and execute
        gen_z3_code = extract_python_code(generated_text)
        if gen_z3_code:
            ans, err = execute_z3_code_with_helpers(gen_z3_code)
        else:
            ans, err = "NoCode", "No python code block found"
            
        # Compare
        norm_expected = expected_ans.strip().lower()
        norm_ans = ans.strip().lower()
        if norm_ans in ['unknown', 'uncertain']: norm_ans = 'unknown'
        if norm_expected in ['unknown', 'uncertain']: norm_expected = 'unknown'
        
        total += 1
        if norm_ans == norm_expected and not err:
            correct += 1
        else:
            print(f"\n❌ Sai lệch tại mẫu {total}:")
            print(f"  - Expected: {expected_ans}")
            print(f"  - Actual: {ans}")
            if err:
                print(f"  - Error: {err}")
            print("  - Generated Z3 Code:")
            print("-" * 40)
            print(gen_z3_code)
            print("-" * 40)
            print("  - Expected Z3 Code:")
            print("-" * 40)
            print(z3_code_expected)
            print("-" * 40)
                
    accuracy = (correct / total) * 100 if total > 0 else 0.0
    print("\n" + "="*50)
    print(f"📈 KẾT QUẢ ĐÁNH GIÁ ĐỘ CHÍNH XÁC")
    print(f"  - Tổng số mẫu đánh giá: {total}")
    print(f"  - Số mẫu chính xác: {correct}")
    print(f"  - ĐỘ CHÍNH XÁC ĐẠT ĐƯỢC: {accuracy:.2f}%")
    print("="*50)
    
    # Restore training mode
    model.train()

evaluate_model_accuracy(model, tokenizer, DATA_FILE, num_samples=30)


📊 Đang chạy đánh giá trên 30 mẫu test ngẫu nhiên...


 13%|█▎        | 4/30 [00:51<06:01, 13.92s/it]


❌ Sai lệch tại mẫu 4:
  - Expected: NoOutput
  - Actual: NoOutput
  - Error: File "<string>", line 73
    ans = check_mcq(s, {)
                        ^
SyntaxError: closing parenthesis ')' does not match opening parenthesis '{'
  - Generated Z3 Code:
----------------------------------------
from z3 import *

# Define Z3 variables
Tâm_AccumulatedCredits = Int('Tâm_AccumulatedCredits')
ForeignLanguageStandardMet = Bool('ForeignLanguageStandardMet')
IsSecondYearStudent = Bool('IsSecondYearStudent')
IsAcceleratedProgram = Bool('IsAcceleratedProgram')
NamMeetsLanguageStandard = Bool('NamMeetsLanguageStandard')
TâmEligibleForMentorship = Bool('TâmEligibleForMentorship')

s = Solver()

# Model premises constraints
s.add(Tâm_AccumulatedCredits == 40)
s.add(ForeignLanguageStandardMet == True)
s.add(IsSecondYearStudent == And(Tâm_AccumulatedCredits >= 33, Tâm_AccumulatedCredits < 66))
s.add(IsAcceleratedProgram == False)
s.add(NamMeetsLanguageStandard == False)
s.add(TâmEligibleForMentorship 

 43%|████▎     | 13/30 [02:15<03:03, 10.79s/it]


❌ Sai lệch tại mẫu 13:
  - Expected: NoOutput
  - Actual: NoOutput
  - Error: File "<string>", line 65
    ans = check_mcq(s, {)
                        ^
SyntaxError: closing parenthesis ')' does not match opening parenthesis '{'
  - Generated Z3 Code:
----------------------------------------
from z3 import *

# Define Z3 variables
Student_UnderstandingMaterial = Bool('Student_UnderstandingMaterial')
Student_Revising = Bool('Student_Revising')

s = Solver()

# Model Premises constraints
s.add(Student_UnderstandingMaterial == True)
s.add(Student_Revising == True)

# Map MCQ statements to Z3 properties
OptionA = And(Student_UnderstandingMaterial, Student_Revising)
OptionB = And(Student_Revising, Not(Student_UnderstandingMaterial))
OptionC = Not(Or(Student_UnderstandingMaterial, Student_Revising))
OptionD = Implies(Student_UnderstandingMaterial, Not(Student_Revising))

# Check MCQ using helper
ans = check_mcq(s, {
    'A': OptionA,
    'B': OptionB,
    'C': OptionC,
    'D': OptionD
})

...


❌ Sai lệch tại mẫu 25:
  - Expected: NoOutput
  - Actual: NoOutput
  - Error: File "<string>", line 63
    Implies(Student_Revising == True, Student_Studying == True),
IndentationError: unexpected indent
  - Generated Z3 Code:
----------------------------------------
from z3 import *

# Define Z3 boolean variables
Student_Studying = Bool('Student_Studying')
Student_Revising = Bool('Student_Revising')
Student_AttendingTutorials = Bool('Student_AttendingTutorials')

s = Solver()

# Model Premises constraints
s.add(Student_Studying == True)  # Every student is studying
s.add(Or(Student_Revising == True))  # There exists at least one student who is revising
s.add(Implies(Student_AttendingTutorials == True, Student_Studying == True))  # If there exists at least one student who is attending tutorials, then every student is studying
s.add(Student_AttendingTutorials == True)  # Every student is attending tutorials

# Define the statement to check
Statement = And(
    Implies(Student_Revising 

 90%|█████████ | 27/30 [04:36<00:31, 10.57s/it]


❌ Sai lệch tại mẫu 27:
  - Expected: NoOutput
  - Actual: NoOutput
  - Error: File "<string>", line 71
    ans = check_mcq(s, {)
                        ^
SyntaxError: closing parenthesis ')' does not match opening parenthesis '{'
  - Generated Z3 Code:
----------------------------------------
from z3 import *

# Define Z3 variables
Transparency = Bool('Transparency')
Auditable = Bool('Auditable')
Fairness = Bool('Fairness')
Trustworthy = Bool('Trustworthy')

s = Solver()

# Model Premises constraints
s.add(Implies(Transparency, Auditable))
s.add(Implies(Fairness, Trustworthy))
s.add(Implies(Fairness, Implies(Transparency, Auditable)))

# Model Facts
# No specific facts given, so we don't add any additional constraints here

# Map MCQ statements to Z3 properties
OptionA = Implies(Fairness, Implies(Transparency, Auditable))
OptionB = Not(Implies(Fairness, Implies(Transparency, Auditable)))
OptionC = BoolVal(False)  # Both true and false is not possible in Z3
OptionD = Not(And(Auditable

100%|██████████| 30/30 [05:18<00:00, 10.61s/it]


📈 KẾT QUẢ ĐÁNH GIÁ ĐỘ CHÍNH XÁC
  - Tổng số mẫu đánh giá: 30
  - Số mẫu chính xác: 24
  - ĐỘ CHÍNH XÁC ĐẠT ĐƯỢC: 80.00%


In [15]:
final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"🎉 Adapter đã được lưu an toàn tại: {final_adapter_path}")

🎉 Adapter đã được lưu an toàn tại: ./qwen_2.5_coder_7b_logic_lora/final_adapter


In [16]:
from huggingface_hub import HfApi, login

TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")
LORA_REPO_ID = "kotorii1/qwen2.5-coder-7b-logic-solver-lora"

if not TOKEN:
    print("⚠️ Vui lòng điền mã Token Hugging Face hoặc thiết lập biến môi trường HF_TOKEN để upload model!")
else:
    print("🔑 Đang tiến hành đăng nhập vào Hugging Face...")
    login(token=TOKEN)

    print(f"⚡ Đang khởi tạo kho chứa Adapter '{LORA_REPO_ID}'...")
    api = HfApi(token=TOKEN)
    try:
        api.create_repo(
            repo_id=LORA_REPO_ID,
            repo_type="model",
            private=True,
            exist_ok=True
        )
        print("✅ Repository đã được thiết lập thành công.")
    except Exception as e:
        print(f"⚠️ Cảnh báo khi tạo repo (có thể đã tồn tại): {e}")

    print(f"🚀 Đang đẩy toàn bộ LoRA adapter từ {final_adapter_path} lên Hugging Face...")
    api.upload_folder(
        folder_path=final_adapter_path,
        repo_id=LORA_REPO_ID,
        repo_type="model"
    )
    print(f"🎉 Thành công mỹ mãn! Adapter của bạn đã được lưu trữ tại: https://huggingface.co/{LORA_REPO_ID}")

🔑 Đang tiến hành đăng nhập vào Hugging Face...
⚡ Đang khởi tạo kho chứa Adapter 'kotorii1/qwen2.5-coder-7b-logic-solver-lora'...
✅ Repository đã được thiết lập thành công.
🚀 Đang đẩy toàn bộ LoRA adapter từ ./qwen_2.5_coder_7b_logic_lora/final_adapter lên Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🎉 Thành công mỹ mãn! Adapter của bạn đã được lưu trữ tại: https://huggingface.co/kotorii1/qwen2.5-coder-7b-logic-solver-lora
